# 🧪 bayes-corner — Predictive Model Checking for PyMC

**GSoC 2026 Prototype · Shashank Dubey · HEC Paris**

---

This notebook demonstrates `PredictiveCheck`, a composable wrapper around PyMC that
makes prior and posterior predictive checks a one-liner.

**What we do here:**
1. Install dependencies
2. Clone `bayes-corner` from GitHub and import the library
3. Generate deliberately **misspecified data** — bimodal truth, single-Gaussian model
4. Run prior predictive checks
5. Fit the model and run posterior predictive checks
6. Compute **Bayesian p-values** for multiple test statistics
7. Run a full diagnostic sweep with `checker.summary()`

**Key idea (McLatchie et al., 2025):** A model can have a perfectly converged posterior
yet still be misspecified. Predictive checks — especially scoring-rule-grounded ones —
are the primary tool for detecting this.

## 0 · Setup

Install PyMC and ArviZ, then clone the `bayes-corner` repo so we can import the library.

> **Runtime:** This cell takes ~2 minutes on a fresh Colab instance.

In [ ]:
# Install dependencies
!pip install pymc arviz --quiet

# Clone bayes-corner from GitHub
import os
if not os.path.exists('bayes-corner'):
    !git clone https://github.com/YOUR_USERNAME/bayes-corner.git

# Add to path so we can import bayes_corner
import sys
sys.path.insert(0, 'bayes-corner')

print('✅ Setup complete')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import matplotlib

# The library we are building
from bayes_corner.predictive_check import PredictiveCheck

# Colab display settings
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

print(f'PyMC version  : {pm.__version__}')
print(f'ArviZ version : {az.__version__}')
print(f'NumPy version : {np.__version__}')

## 1 · Generate Misspecified Data

We deliberately create a situation that mirrors real-world model misspecification:

- **True data-generating process:** a 50/50 mixture of N(-2, 0.5) and N(2, 0.5)  
- **Model we will fit:** a single Gaussian N(μ, σ)

This is structurally identical to the **non-trivial misspecification** regime in
McLatchie et al. (2025): no single Gaussian parameter can describe the bimodal truth,
but a *mixture* of Gaussians (a convex combination over the model class) can recover
it exactly.

The Bayesian posterior will concentrate confidently on μ ≈ 0 — which is wrong in a
deep sense. The predictive check is the only thing that reveals this.

In [ ]:
rng = np.random.default_rng(42)
n = 300

# True DGP: bimodal
data = np.concatenate([
    rng.normal(-2, 0.5, n // 2),   # left mode
    rng.normal( 2, 0.5, n // 2),   # right mode
])

print(f'n = {len(data)}')
print(f'mean  = {data.mean():.3f}  (true: 0.0)')
print(f'std   = {data.std():.3f}  (true: ~2.06 — inflated by bimodality)')
print(f'True distribution: 0.5 × N(-2, 0.5) + 0.5 × N(2, 0.5)')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(data, bins=40, color='steelblue', alpha=0.7, density=True)
ax.set_title('Observed Data — bimodal truth', fontsize=13)
ax.set_xlabel('y')
ax.set_ylabel('Density')
plt.tight_layout()
plt.show()

## 2 · Prior Predictive Check

Before fitting, we ask: **are the priors compatible with the scale of the data?**

This is a sanity check, not a model selection criterion. Priors that generate
predictions in completely implausible ranges (e.g., μ ~ N(0, 1000)) will produce
a prior predictive distribution that barely overlaps the observed data.

We define our misspecified model: a single Gaussian with weakly informative priors.

In [ ]:
with pm.Model() as gaussian_model:
    # Weakly informative priors
    mu    = pm.Normal('mu',    mu=0,    sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=3)

    # Likelihood — single Gaussian (MISSPECIFIED: truth is bimodal)
    obs = pm.Normal('obs', mu=mu, sigma=sigma, observed=data)

print(gaussian_model.model)
pm.model_to_graphviz(gaussian_model)

In [ ]:
# Wrap the model — no fitting needed yet for prior check
# We pass idata=None and let PredictiveCheck handle prior sampling
with gaussian_model:
    idata_prior_only = pm.sample_prior_predictive(random_seed=42)

checker_prior = PredictiveCheck(
    model=gaussian_model,
    idata=idata_prior_only,
    observed=data,
    var_name='obs',
    random_seed=42,
)

fig = checker_prior.prior_check(n_samples=50)
plt.show()

print()
print('Interpretation:')
print('  Left panel : each faint blue histogram is one draw from the prior predictive.')
print('  Right panel: ArviZ plot_ppc view of the same samples.')
print('  The black step curve is the observed data.')
print()
print('  ✅ The prior predictive broadly covers the data range — priors are not')
print('  wildly misspecified. But notice it covers a very wide range, which is')
print('  fine for a weakly informative prior.')

## 3 · Fit the Model

Now we sample the posterior with NUTS. The model will converge — diagnostics will
look fine (R̂ ≈ 1, good ESS). This is the trap: **convergence does not mean the
model is correct**. The posterior predictive check will reveal the problem.

In [ ]:
with gaussian_model:
    idata = pm.sample(
        draws=1000,
        tune=1000,
        random_seed=42,
        progressbar=True,
    )

# Convergence diagnostics
az.plot_trace(idata, var_names=['mu', 'sigma'], compact=True)
plt.tight_layout()
plt.show()

print()
summary = az.summary(idata, var_names=['mu', 'sigma'])
print(summary)
print()
print('Notice:')
print(f'  μ posterior mean ≈ {idata.posterior["mu"].values.mean():.3f}')
print('  R̂ close to 1 — chains look converged.')
print('  But the true μ is not 0. The model is structurally wrong.')

## 4 · Posterior Predictive Check

Now we ask: **does the fitted model reproduce the observed data?**

The `PredictiveCheck` wrapper:
1. Draws samples from the posterior predictive distribution (lazy, cached)
2. Overlays them against the observed data
3. Calls ArviZ's `plot_ppc` for the standard view

All in one line.

In [ ]:
checker = PredictiveCheck(
    model=gaussian_model,
    idata=idata,
    observed=data,
    var_name='obs',
    random_seed=42,
)

fig = checker.posterior_check(n_samples=50)
plt.show()

print()
print('Interpretation:')
print('  Each faint orange histogram is one draw from the posterior predictive.')
print('  The black step curve is the observed bimodal data.')
print()
print('  ❌ The posterior predictive is unimodal — centred near 0.')
print('  It completely misses the two modes at -2 and +2.')
print('  The model is misspecified, and the check catches it visually.')

## 5 · Formal Bayesian p-values

Visual checks are powerful but subjective. The `score()` method formalises
the check using **Bayesian p-values**:

$$p_B = P\left(T(y^{\text{rep}}) \geq T(y^{\text{obs}})\right)$$

where $T$ is a test statistic and $y^{\text{rep}}$ are posterior predictive draws.

**Interpretation:**
- $p_B \approx 0.5$ → model reproduces this feature well
- $p_B < 0.05$ or $p_B > 0.95$ → model **fails** to reproduce this feature

**Key insight:** The `mean` p-value will look fine (~0.5) because the posterior
correctly estimates the mean ≈ 0. But `kurtosis` and `std` will be extreme — 
flagging the bimodal structure the model cannot capture.

In [ ]:
# Mean — should look fine (model gets the mean right)
result_mean = checker.score(stat='mean')
plt.show()

p = result_mean['bayesian_p_value']
print(f"Bayesian p-value (mean) = {p:.3f}")
print(f"Observed mean = {result_mean['observed_value']:.3f}")
print()
if 0.05 < p < 0.95:
    print('✅ Model reproduces the mean well. Expected — the Gaussian centred on 0.')
else:
    print('❌ Model fails to reproduce the mean.')

In [ ]:
# Std — will be off because bimodal spread > Gaussian spread
result_std = checker.score(stat='std')
plt.show()

p = result_std['bayesian_p_value']
print(f"Bayesian p-value (std) = {p:.3f}")
print(f"Observed std = {result_std['observed_value']:.3f}")
print()
if p < 0.05 or p > 0.95:
    print('❌ Model FAILS to reproduce the standard deviation.')
    print('   The bimodal data has a wider spread than any single Gaussian can match.')
else:
    print('✅ Model reproduces the std.')

In [ ]:
# Kurtosis — the most sensitive statistic for detecting bimodality
# Bimodal distributions have very different kurtosis than a Gaussian
result_kurt = checker.score(stat='kurtosis')
plt.show()

p = result_kurt['bayesian_p_value']
print(f"Bayesian p-value (kurtosis) = {p:.3f}")
print(f"Observed kurtosis = {result_kurt['observed_value']:.3f}")
print()
if p < 0.05 or p > 0.95:
    print('❌ Model FAILS to reproduce the kurtosis.')
    print('   This is the clearest signal of the bimodal misspecification.')
    print('   A bimodal distribution has negative excess kurtosis (platykurtic),')
    print('   while a Gaussian has kurtosis = 0 by definition.')
else:
    print('✅ Model reproduces the kurtosis.')

## 6 · Custom Test Statistic

You can pass any function as a test statistic. Here we test for bimodality
directly using **Hartigan's Dip statistic** approximation — the proportion
of data in the central trough between the two modes.

In [ ]:
def trough_mass(y):
    """Proportion of data in the trough region [-0.5, 0.5].
    Bimodal data has low mass here; a Gaussian centred on 0 has high mass.
    """
    return float(np.mean((y > -0.5) & (y < 0.5)))

result_custom = checker.score(
    stat='custom',
    fn=trough_mass,
    group='posterior',
)
plt.show()

p = result_custom['bayesian_p_value']
print(f"Bayesian p-value (trough mass) = {p:.3f}")
print(f"Observed trough mass = {result_custom['observed_value']:.3f}")
print()
print('The observed data has very little mass near 0 (the trough between modes).')
print('The Gaussian model predicts lots of mass near 0 — this is the mismatch.')
if p < 0.05:
    print('❌ p < 0.05: model badly overestimates mass in the trough.')

## 7 · Full Diagnostic Sweep

`checker.summary()` runs all built-in statistics at once and returns a
dict of Bayesian p-values — useful for a quick model health check.

In [ ]:
import pandas as pd

summary = checker.summary(group='posterior')

# Format as a table
rows = []
for stat, p in summary.items():
    if p is None:
        flag = 'ERROR'
    elif p < 0.05 or p > 0.95:
        flag = '❌ FAIL'
    elif p < 0.1 or p > 0.9:
        flag = '⚠️  WARN'
    else:
        flag = '✅ OK'
    rows.append({'Statistic': stat, 'Bayesian p-value': round(p, 3) if p else None, 'Verdict': flag})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print('Model verdict: MISSPECIFIED')
print('Statistics that fail reveal the structure of the misspecification:')
print('  - std/kurtosis failures → the model misses the bimodal spread')
print('  - mean OK → the model correctly centres on 0 (the grand mean)')

## 8 · Compare: Well-Specified Model

As a sanity check, we fit the *correct* model — a two-component
Gaussian mixture — and confirm that its predictive checks all pass.

> This mirrors the **convex recovery** regime from McLatchie et al. (2025):
> the true DGP is a convex combination of elements from the model class,
> so the posterior predictive can recover it exactly.

In [ ]:
import pytensor.tensor as pt

with pm.Model() as mixture_model:
    # Mixture weight
    w = pm.Dirichlet('w', a=np.ones(2))

    # Component means and shared std
    mu_comps = pm.Normal('mu_comps', mu=0, sigma=3, shape=2)
    sigma_mix = pm.HalfNormal('sigma_mix', sigma=1)

    # Mixture likelihood
    obs_mix = pm.Mixture(
        'obs_mix',
        w=w,
        comp_dists=pm.Normal.dist(mu=mu_comps, sigma=sigma_mix, shape=(2,)),
        observed=data,
    )

with mixture_model:
    idata_mix = pm.sample(draws=1000, tune=1000, random_seed=42,
                          progressbar=True, target_accept=0.9)

print('\nMixture model posterior summary:')
print(az.summary(idata_mix, var_names=['w', 'mu_comps', 'sigma_mix']))

In [ ]:
checker_mix = PredictiveCheck(
    model=mixture_model,
    idata=idata_mix,
    observed=data,
    var_name='obs_mix',
    random_seed=42,
)

fig = checker_mix.posterior_check(n_samples=50)
plt.show()

print('\nFull sweep — mixture model (should all be ✅):')
summary_mix = checker_mix.summary(group='posterior')
rows = []
for stat, p in summary_mix.items():
    if p is None:
        flag = 'ERROR'
    elif p < 0.05 or p > 0.95:
        flag = '❌ FAIL'
    elif p < 0.1 or p > 0.9:
        flag = '⚠️  WARN'
    else:
        flag = '✅ OK'
    rows.append({'Statistic': stat, 'Bayesian p-value': round(p, 3) if p else None, 'Verdict': flag})

import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))

## 9 · Key Takeaways

| | Single Gaussian (wrong) | Mixture (right) |
|---|---|---|
| R̂ | ✅ ≈ 1 | ✅ ≈ 1 |
| Posterior predictive visual | ❌ Unimodal | ✅ Bimodal |
| p-value (mean) | ✅ ~0.5 | ✅ ~0.5 |
| p-value (std) | ❌ Extreme | ✅ ~0.5 |
| p-value (kurtosis) | ❌ Extreme | ✅ ~0.5 |

**The lesson:** Convergence diagnostics alone cannot detect model misspecification.
Predictive checks are the only tool that reveals whether the model actually 
describes the data — not just whether NUTS sampled efficiently from it.

---

### What `bayes-corner` adds

Without this library, the workflow above requires:
- Manual calls to `pm.sample_prior_predictive` / `pm.sample_posterior_predictive`
- Manual xarray extraction from InferenceData
- Custom plotting loops
- Custom test statistic loops

With `PredictiveCheck`, everything from Section 4 onwards is **one method call**.

---

### References

- McLatchie, Y., Chérief-Abdellatif, B-E., Frazier, D.T., & Knoblauch, J. (2025).  
  *Predictively Oriented Posteriors.* arXiv:2510.01915.
- Gelman, A., et al. (2020). *Bayesian Workflow.* arXiv:2011.01808.
- Gabry, J., et al. (2019). *Visualization in Bayesian workflow.* JRSS-A, 182(2).